### Load environment variables

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("API key is configured.")
else:
    raise ValueError("OPENAI_API_KEY is missing from .env")

API key is configured.


### Initialize the LLM

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5-nano",
    temperature=0
)

/Users/chethanprabhas/CLAUDECODE/RAG Tutorial/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


### Extract text from the PDF

In [5]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("./Docs/ABC_Bank_Customer_Support_FAQ_Sample.pdf")

docs = loader.load()

docs

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-11T17:30:12+00:00', 'author': 'ABC Bank (Fictional Sample)', 'keywords': '', 'moddate': '2026-10-11T17:30:12+00:00', 'subject': '(unspecified)', 'title': 'ABC Bank Customer Support FAQ - Sample', 'trapped': '/False', 'source': './Docs/ABC_Bank_Customer_Support_FAQ_Sample.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='FICTIONAL SAMPLE \x7f FOR DEMONSTRATION / TESTING ONLY\nPage 1\nABC BANK\nCUSTOMER SUPPORT \x7f FREQUENTLY ASKED QUESTIONS\nDEMO DOCUMENT   All contact details, policies, amounts and addresses below are fictional sample values. Do not use for\nreal banking.\n1. Frequently Asked Questions\nHow do I open a savings account?\nComplete the sample account-opening form and provide the requested identity and address documents. For this demo, the\nreview target is 1–2 business days. Actual requirements and timelines vary by bank.\nHow ca

### Add metadata

In [6]:
for i in docs:
    i.metadata = {
        "source": "ABC_Bank_Customer_Support_FAQ_Sample.pdf",
        "developer": "Chethan Prabhas",
    }

### Split the PDF into chunks

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'ABC_Bank_Customer_Support_FAQ_Sample.pdf', 'developer': 'Chethan Prabhas'}, page_content='FICTIONAL SAMPLE \x7f FOR DEMONSTRATION / TESTING ONLY\nPage 1\nABC BANK\nCUSTOMER SUPPORT \x7f FREQUENTLY ASKED QUESTIONS\nDEMO DOCUMENT   All contact details, policies, amounts and addresses below are fictional sample values. Do not use for\nreal banking.\n1. Frequently Asked Questions\nHow do I open a savings account?\nComplete the sample account-opening form and provide the requested identity and address documents. For this demo, the\nreview target is 1–2 business days. Actual requirements and timelines vary by bank.\nHow can I check my account balance?\nUse the bank’s verified mobile app, internet banking portal, ATM, or a branch. Never share your password, PIN, or one-time\npassword (OTP) with anyone.\nWhat should I do if my debit card is lost or stolen?\nImmediately lock the card through the official banking app or contact the verified support number. Ask supp

### Create embeddings

In [8]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

### Create the vector store with Chroma

In [9]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

### Perform semantic search

In [11]:
results = vectorstore.similarity_search(
    "what is the phone number of ABC Bank in Bangalore?",
    k=2
)

for index, doc in enumerate(results, start=1):
    print(f"\n--- Result {index} ---")
    print(doc.page_content)
    print(doc.metadata)


--- Result 1 ---
FICTIONAL SAMPLE  FOR DEMONSTRATION / TESTING ONLY
Page 2
ABC BANK
BRANCH DIRECTORY & SERVICE GUIDELINES
3. Sample Branch Addresses — Four Major Cities
The branch names and street addresses below are invented for sample documentation. They are not verified real-world ABC
Bank locations.
BENGALURU — ABC Bank – Bengaluru Demo Branch
No. 24, Sample Tech Park Road, Demo Layout
Bengaluru, Karnataka 560001
Phone: +91 80000 10001
HYDERABAD — ABC Bank – Hyderabad Demo Branch
Plot 18, Example Financial District, Mock Hills
Hyderabad, Telangana 500001
Phone: +91 80000 10002
CHENNAI — ABC Bank – Chennai Demo Branch
No. 7, Placeholder Marina Avenue, Test Nagar
Chennai, Tamil Nadu 600001
Phone: +91 80000 10003
MUMBAI — ABC Bank – Mumbai Demo Branch
Office 12, Fictional Market Street, Sample District
Mumbai, Maharashtra 400001
Phone: +91 80000 10004
4. Service Policies (Sample Values)
Branch hours
Monday–Friday: 9:30 AM–4:00 PM; Saturday: 10:00 AM–1:00 PM; Sunday and public holida

### Pass retrieved context to the LLM

In [12]:
context = vectorstore.similarity_search(
    "what is the phone number of ABC Bank in Bangalore?",
    k=2
)

response = llm.invoke(
    f"what is the phone number of ABC Bank in Bangalore? You can answer using the following context: {context}"
)

print(response.content)

The Bengaluru (Bangalore) branch phone number is +91 80000 10001. (Note: This is from a fictional sample document.)


In [16]:
context = vectorstore.similarity_search(
    "what is the phone number of ABC Bank in Bangalore?",
    k=2
)

response = llm.invoke(
    f"what is the phone number of ICICIC Bank in Bangalore? You can answer using the following context: {context}"
)

print(response.content)

The provided document does not list ICICIC/ICICI Bank at all. It shows a fictional sample for ABC Bank’s Bengaluru Demo Branch with the phone number +91 80000 10001. Since this is a fictional sample, it’s not a real contact for ICICI Bank. If you need ICICI Bank in Bangalore, please share a document that contains ICICI Bank details or check ICICI Bank’s official site or directory.
